[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/signal38/signal38.github.io/blob/main/notebooks/04_export_onnx.ipynb)

# Notebook 04 — Merge & HuggingFace Hub

Merges the LoRA adapter into LFM2-350M and pushes the merged model to HuggingFace Hub.

> **Note:** LFM2's hybrid architecture is not yet supported by `optimum`'s ONNX exporter or transformers.js.
> Browser inference is deferred — the landing page serves pre-computed results from `03_evaluate.ipynb`.

**Prerequisites:** Run `02_finetune.ipynb` first to publish the LoRA adapter.

**Requirements:** T4 GPU, `GITHUB_TOKEN` and `HF_TOKEN` Colab secrets.

**Outputs:**
- `signal38/lfm2-nk-risk` — merged 16-bit model on HuggingFace Hub


In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('04_export_onnx', requirements_path=str(REPO / 'requirements.txt'))
# Runtime may restart after this cell.

In [ ]:
import subprocess, sys, os, json, re
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import prepare_notebook, require_local_adapter
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)

# Verify prerequisites
require_local_adapter(REPO)  # raises if adapter not published

print('Prerequisites satisfied.')

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Missing HF_TOKEN Colab secret. Add it via the key icon in the Colab left sidebar.')

login(token=hf_token)
api = HfApi()

REPO_ID = 'signal38/lfm2-nk-risk'
print(f'Publishing to: {REPO_ID}')

api.create_repo(repo_id=REPO_ID, private=False, exist_ok=True)
print(f'Hub repo ready: {REPO_ID}')

api.upload_folder(
    folder_path=str(merged_dir),
    repo_id=REPO_ID,
    repo_type='model',
    commit_message='Add merged LFM2-350M NK risk model',
)
print(f'Pushed to https://huggingface.co/{REPO_ID}')


In [ ]:
# Smoke test — load from Hub and run one inference
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer_hub = AutoTokenizer.from_pretrained(REPO_ID)
model_hub = AutoModelForCausalLM.from_pretrained(
    REPO_ID, torch_dtype=torch.float16, device_map='auto'
)

test_prompt = 'Analyze NK military activity with high Goldstein negativity. Respond in JSON.'
inputs = tokenizer_hub(test_prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model_hub.generate(**inputs, max_new_tokens=64)
print(tokenizer_hub.decode(out[0], skip_special_tokens=True)[:300])
print('Smoke test passed.')
